# PN21 — ridge-straddling two-child retention

## tl;dr

The last prime gate below `sqrt(n)` and first above it retained effectively **0%** of the exact parent factor-progress coordinate on the held-out half (`R²=-0.0000292`). Prime-ridge AUCs were `0.4997` and `0.4991`. This is a development null for that child definition; the sealed 87-bit target was not opened.


## Context & Methods

PN21 asks whether a genuinely ridge-straddling immediate pair provides a TheFormula-like dominant component. The full parent is `1` for primes and `2 log(lpf(n))/log(n)` for composites. The two children are raw residue phases at the nearest prime gates immediately below and above `sqrt(n)`.

### Key Assumptions

- The square-root boundary is the candidate rung boundary.
- A fixed 32×32 raw-phase grid can detect information retained by the pair without claiming that the grid is an ARA formula.
- The first half of the opened interval trains the grid; the second half tests retention.


In [1]:
import json
import math
from pathlib import Path
import numpy as np
from pn21_ridge_straddling_two_child import (
    LOW, HIGH, MID, prime_table, segmented_least_prime_factor,
    phase_pair, pair_diagnostics,
)

HERE = Path.cwd()
saved = json.loads((HERE / 'PN21_RIDGE_STRADDLING_TWO_CHILD_RESULTS.json').read_text(encoding='utf-8'))
validation = json.loads((HERE / 'PN21_RIDGE_STRADDLING_TWO_CHILD_VALIDATION.json').read_text(encoding='utf-8'))
print('Loaded frozen PN21 outputs and independent validation.')


Loaded frozen PN21 outputs and independent validation.


## Data

The complete opened interval contains one million raw integers. Analysis is restricted to its 500,000 odd candidates. The parent coordinate is reconstructed exactly from a segmented least-factor sieve.


In [2]:
primes = prime_table(math.isqrt(HIGH - 1) + 200)
least_all = segmented_least_prime_factor(LOW, HIGH, primes)
all_numbers = np.arange(LOW, HIGH, dtype=np.int64)
odd = (all_numbers & 1) == 1
numbers = all_numbers[odd]
least = least_all[odd]
labels = least == 0
parent = np.ones(numbers.size, dtype=np.float64)
composite = ~labels
parent[composite] = 2.0 * np.log(least[composite]) / np.log(numbers[composite])
print('odd candidates:', numbers.size)
print('primes:', int(labels.sum()), 'composites:', int(composite.sum()))
assert numbers.size == 500_000
assert int(labels.sum()) == 45_166


odd candidates: 500000
primes: 45166 composites: 454834


## Results

The same computation is run for the ridge-straddling pair and the two-below-ridge control.


In [3]:
root = np.floor(np.sqrt(numbers)).astype(np.int64)
position = np.searchsorted(primes, root, side='right')
q_minus = primes[position - 1]
q_plus = primes[position]
q_second_minus = primes[position - 2]
straddle_a, straddle_b = phase_pair(numbers, q_minus, q_plus)
same_a, same_b = phase_pair(numbers, q_minus, q_second_minus)
train = numbers < MID
test = ~train
straddling = pair_diagnostics('straddling', numbers, parent, labels, straddle_a, straddle_b, train, test)
same_side = pair_diagnostics('same-side', numbers, parent, labels, same_a, same_b, train, test)
for result in (straddling, same_side):
    print(result['name'])
    print('  held-out R2:', result['retention']['heldout_retained_r2'])
    print('  parent correlation:', result['closure_summary']['pearson_with_full_parent'])
    print('  joint AUC:', result['prime_diagnostics']['joint_ridge_auc'])
    print('  closure AUC:', result['prime_diagnostics']['closure_ridge_auc'])


straddling
  held-out R2: -2.9207029406563834e-05
  parent correlation: 0.00039163725901819737
  joint AUC: 0.4996984112001532
  closure AUC: 0.4991371963682423
same-side
  held-out R2: -2.2956775237670257e-05
  parent correlation: -0.00023444843782039726
  joint AUC: 0.49968351150115137
  closure AUC: 0.5006996063526248


In [4]:
assert abs(straddling['retention']['heldout_retained_r2'] - saved['straddling_pair']['retention']['heldout_retained_r2']) < 1e-12
assert abs(same_side['retention']['heldout_retained_r2'] - saved['same_side_control']['retention']['heldout_retained_r2']) < 1e-12
assert straddling['retention']['heldout_retained_r2'] < 0.90
assert straddling['retention']['heldout_retained_r2'] <= same_side['retention']['heldout_retained_r2']
print('Frozen threshold: FAIL')
print('Independent validation:', validation['status'], validation['checks_passed'], '/', validation['checks_total'])
assert validation['status'] == 'PASS'


Frozen threshold: FAIL
Independent validation: PASS 12 / 12


## Takeaways

1. Straddling the square-root ridge fixes orientation but does not make the gates endogenous children of the integer.
2. The pair retained none of the full parent variance out of sample and did not rank primes above chance.
3. The next candidate decomposition should follow actual collision/survival state through the sieve web, potentially including temporal change across neighboring integers.
4. No blind prime-location test is justified from PN21; the 87-bit anchor remains sealed.
